In [1]:
import os
%pwd

'c:\\Users\\gimep\\Downloads\\workspace\\text_summarizer\\research'

In [2]:
os.chdir("../")

In [3]:
%pwd

'c:\\Users\\gimep\\Downloads\\workspace\\text_summarizer'

In [4]:
# Add these imports at the top
import gc
import torch
import os
from dataclasses import dataclass
from pathlib import Path
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from datasets import load_from_disk
from src.textsummarizer.constants import *
from src.textsummarizer.utils.common import read_yaml, create_directories

c:\Users\gimep\Downloads\workspace\text_summarizer\.textenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: str  # MUST be string for Hugging Face models
    num_train_epochs: int
    warmup_steps: int
    per_device_train_batch_size: int
    weight_decay: float
    logging_steps: int
    evaluation_strategy: str
    eval_steps: int
    save_steps: int
    gradient_accumulation_steps: int


In [6]:
from src.textsummarizer.constants import *
from src.textsummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def _safe_convert_to_int(self, value):
        if isinstance(value, (int, float)):
            return int(value)
        elif isinstance(value, str):
            if 'e' in value.lower():
                return int(float(value))
            else:
                return int(float(value))
        else:
            return int(value)

    def _safe_convert_to_float(self, value):
        if isinstance(value, (int, float)):
            return float(value)
        elif isinstance(value, str):
            return float(value)
        else:
            return float(value)

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        params = self.params.TrainingArguments

        create_directories([config.root_dir])

        # FIX: Convert model_ckpt to proper Hugging Face format
        model_ckpt = str(config.model_ckpt)
        # Replace backslashes with forward slashes for Hugging Face models
        if '\\' in model_ckpt:
            model_ckpt = model_ckpt.replace('\\', '/')
            print(f"Fixed model path: {model_ckpt}")

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            model_ckpt=model_ckpt,  # Use the fixed string
            num_train_epochs=self._safe_convert_to_int(params.num_train_epochs),
            warmup_steps=self._safe_convert_to_int(params.warmup_steps),
            per_device_train_batch_size=self._safe_convert_to_int(params.per_device_train_batch_size),
            weight_decay=self._safe_convert_to_float(params.weight_decay),
            logging_steps=self._safe_convert_to_int(params.logging_steps),
            evaluation_strategy=str(params.evaluation_strategy),
            eval_steps=self._safe_convert_to_int(params.eval_steps),
            save_steps=self._safe_convert_to_int(params.save_steps),
            gradient_accumulation_steps=self._safe_convert_to_int(params.gradient_accumulation_steps)
        )
        return model_trainer_config


In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
import torch
from datasets import load_from_disk

In [9]:
# Update the ModelTrainer class
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus, padding=True)

        # Loading the data
        dataset_samsum_pt = load_from_disk(self.config.data_path)
        
        # Create trainer with smaller subset for testing
        train_dataset = dataset_samsum_pt["train"].select(range(10))
        eval_dataset = dataset_samsum_pt["validation"].select(range(2))

        # Fix tokenizer settings
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        if tokenizer.bos_token is None:
            tokenizer.bos_token = tokenizer.eos_token
        
        # Update model config to match tokenizer
        model_pegasus.config.pad_token_id = tokenizer.pad_token_id
        model_pegasus.config.eos_token_id = tokenizer.eos_token_id
        model_pegasus.config.bos_token_id = tokenizer.bos_token_id

        # Training arguments with fixes
        training_args = TrainingArguments(
            output_dir=os.path.join(self.config.root_dir, "pegasus-samsum"),
            num_train_epochs=self.config.num_train_epochs,  # Use config value
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_train_batch_size,
            warmup_steps=self.config.warmup_steps,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.evaluation_strategy,  # Correct parameter name
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            fp16=torch.cuda.is_available(),
            dataloader_pin_memory=False,
            report_to="none",
            logging_dir=os.path.join(self.config.root_dir, "logs"),
            remove_unused_columns=True,
            load_best_model_at_end=False,
        )

        # Create trainer with correct parameters
        trainer = Trainer(
            model=model_pegasus,
            args=training_args,
            data_collator=seq2seq_data_collator,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            processing_class=tokenizer,  # Fixed: use tokenizer instead of processing_class
        )
        
        trainer.train()

        # Save model
        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))
        # Save tokenizer
        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "tokenizer"))


In [10]:
# Debug function to check configuration
def debug_config():
    """Debug the configuration loading"""
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    
    print("=== Configuration Debug ===")
    print(f"root_dir: {model_trainer_config.root_dir}")
    print(f"data_path: {model_trainer_config.data_path}")
    print(f"model_ckpt: {model_trainer_config.model_ckpt}")
    print(f"num_train_epochs: {model_trainer_config.num_train_epochs} (type: {type(model_trainer_config.num_train_epochs)})")
    print(f"warmup_steps: {model_trainer_config.warmup_steps} (type: {type(model_trainer_config.warmup_steps)})")
    print(f"save_steps: {model_trainer_config.save_steps} (type: {type(model_trainer_config.save_steps)})")
    print("============================")
    
    return model_trainer_config

!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

In [ ]:
# Add these imports at the top
import gc
import torch

# Clear everything and restart
def full_clean_restart():
    """Complete cleanup and restart training"""
    # Clear all caches
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    print("Cache cleared successfully")
    
    # Debug configuration first
    print("Loading configuration...")
    model_trainer_config = debug_config()
    
    # Reinitialize trainer
    model_trainer = ModelTrainer(config=model_trainer_config)
    return model_trainer



: 

In [ ]:
# Run the complete cleanup and training
print("Starting complete cleanup and training...")
try:
    model_trainer = full_clean_restart()
    model_trainer.train()
except Exception as e:
    print(f"Error during training: {e}")
    # Force cleanup on error
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

Starting complete cleanup and training...
Cache cleared successfully
Loading configuration...
=== Configuration Debug ===
root_dir: artifacts\model_trainer
data_path: artifacts\data_transformation\samsum_dataset
model_ckpt: google/pegasus-cnn_dailymail
num_train_epochs: 1 (type: <class 'int'>)
warmup_steps: 100 (type: <class 'int'>)
save_steps: 300 (type: <class 'int'>)


--- Logging error ---
Traceback (most recent call last):
  File "C:\Python311\Lib\logging\__init__.py", line 1110, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\logging\__init__.py", line 953, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\logging\__init__.py", line 690, in format
    s = self.formatMessage(record)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\logging\__init__.py", line 659, in formatMessage
    return self._style.format(record)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\logging\__init__.py", line 449, in format
    return self._format(record)
           ^^^^^^^^^^^^^^^^^^^^
  File "C:\Python311\Lib\logging\__init__.py", line 445, in _format
    return self._fmt % values
           ~~~~~~~~~~^~~~~~~~
ValueError: unsupported format character ':' (0x3a) at index 26
Call stack:
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<f